# ECG Project — M0: Measurement and Split Preparation

## Purpose
This notebook establishes the initial baseline under the revised evaluation protocol.

It will:
1. Audit class counts by record and person.
2. Generate five repeats of person-grouped five-fold cross-validation.
3. Verify that no person crosses training and validation folds.
4. Train the account-assigned baseline experiment.
5. Calculate pooled-repeat metrics and record-level confidence intervals.
6. Prepare a demo-person impact table for review.

## Account assignment
- Account A: REFERENCE_MODEL
- Account B: BASE_RAW_402

## Evaluation protocol
- 5 repeats × 5 folds.
- Person_ID is the grouping variable.
- All five validation folds are pooled within each repeat.
- Headline metric: average of the five pooled 5-class Macro-F1 scores.
- Also report 4-class Macro-F1 (N, S, V, Q), per-class precision/recall/F1, balanced accuracy, accuracy and confusion matrices.
- Confidence intervals resample records, not beats or folds.

## Scope boundary
This notebook uses all 45 people. It does not exclude demo people or regenerate the final development splits. Demo-person exclusion requires review and explicit approval.

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support
)

# Account A: "A_REFERENCE_MODEL"
# Account B: "B_BASE_RAW_402"
ACCOUNT_ROLE = "A_REFERENCE_MODEL"

SEED = 42
N_SPLITS = 5
N_REPEATS = 5

CLASS_ORDER = ["N", "S", "V", "F", "Q"]
FOUR_CLASS_ORDER = ["N", "S", "V", "Q"]

EXPECTED_COUNTS = {
    "N": 90341,
    "S": 2781,
    "V": 7229,
    "F": 802,
    "Q": 3894
}

EXPECTED_BEATS = 105047
EXPECTED_RECORDS = 46
EXPECTED_PEOPLE = 45

BOOTSTRAP_SAMPLES = 1000
BOOTSTRAP_SEED = 20261105

if ACCOUNT_ROLE not in {
    "A_REFERENCE_MODEL",
    "B_BASE_RAW_402"
}:
    raise ValueError("Invalid ACCOUNT_ROLE.")

OUTPUT_DIR = Path(
    "/kaggle/working/ECG_Project/01_M0_Measurement"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Account:", ACCOUNT_ROLE)
print("Output directory:", OUTPUT_DIR)

Account: A_REFERENCE_MODEL
Output directory: /kaggle/working/ECG_Project/01_M0_Measurement


In [2]:
SIGNAL_COLS = (
    [f"MLII_{i}" for i in range(201)]
    + [f"Lead2_{i}" for i in range(201)]
)

AAMI_MAP = {
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V",
    "F": "F",
    "/": "Q", "f": "Q", "Q": "Q"
}

validated_files = list(
    Path("/kaggle/input").rglob("validated_aami_dataset.csv")
)

processed_files = list(
    Path("/kaggle/input").rglob(
        "MITBIH_MLII_DualLead_Dataset.csv"
    )
)

if validated_files:
    data_path = sorted(
        validated_files,
        key=lambda p: len(str(p))
    )[0]

    data = pd.read_csv(data_path, low_memory=False)

    print("Using validated dataset:", data_path)

    if "AAMI_Label" not in data.columns:
        data["AAMI_Label"] = (
            data["Label"].astype(str).map(AAMI_MAP)
        )

    if "Person_ID" not in data.columns:
        ids = pd.to_numeric(
            data["Record_ID"], errors="raise"
        ).astype(int)

        data["Person_ID"] = [
            "person_201_202"
            if i in (201, 202)
            else f"person_{i}"
            for i in ids
        ]

    if "Row_Index" not in data.columns:
        data["Row_Index"] = np.arange(len(data))

    aami_df = data[
        data["AAMI_Label"].isin(CLASS_ORDER)
    ].copy().reset_index(drop=True)

else:
    if not processed_files:
        raise FileNotFoundError(
            "Could not find the validated dataset or "
            "processed MIT-BIH CSV."
        )

    data_path = sorted(
        processed_files,
        key=lambda p: len(str(p))
    )[0]

    data = pd.read_csv(data_path, low_memory=False)

    print("Using processed dataset:", data_path)

    data["AAMI_Label"] = (
        data["Label"].astype(str).map(AAMI_MAP)
    )

    aami_df = data[
        data["AAMI_Label"].isin(CLASS_ORDER)
    ].copy().reset_index(drop=True)

    aami_df["Row_Index"] = np.arange(len(aami_df))

    ids = pd.to_numeric(
        aami_df["Record_ID"], errors="raise"
    ).astype(int)

    aami_df["Person_ID"] = [
        "person_201_202"
        if i in (201, 202)
        else f"person_{i}"
        for i in ids
    ]

aami_df["Record_ID"] = pd.to_numeric(
    aami_df["Record_ID"], errors="raise"
).astype(int)

aami_df["Row_Index"] = pd.to_numeric(
    aami_df["Row_Index"], errors="raise"
).astype(np.int64)

aami_df["Person_ID"] = aami_df["Person_ID"].astype(str)

assert len(aami_df) == EXPECTED_BEATS
assert aami_df["Record_ID"].nunique() == EXPECTED_RECORDS
assert aami_df["Person_ID"].nunique() == EXPECTED_PEOPLE
assert aami_df["Row_Index"].is_unique

observed_counts = (
    aami_df["AAMI_Label"]
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
    .to_dict()
)

assert observed_counts == EXPECTED_COUNTS, observed_counts

person_201 = aami_df.loc[
    aami_df["Record_ID"] == 201, "Person_ID"
].unique()

person_202 = aami_df.loc[
    aami_df["Record_ID"] == 202, "Person_ID"
].unique()

assert len(person_201) == 1
assert len(person_202) == 1
assert person_201[0] == person_202[0]

print("Valid beats:", len(aami_df))
print("Records:", aami_df["Record_ID"].nunique())
print("People:", aami_df["Person_ID"].nunique())
print("Class counts:", observed_counts)
print("Records 201 and 202 share:", person_201[0])

Using processed dataset: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv
Valid beats: 105047
Records: 46
People: 45
Class counts: {'N': 90341, 'S': 2781, 'V': 7229, 'F': 802, 'Q': 3894}
Records 201 and 202 share: person_201_202


In [3]:
record_counts = (
    aami_df.groupby(
        ["Record_ID", "Person_ID", "AAMI_Label"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_ORDER, fill_value=0)
    .reset_index()
    .sort_values("Record_ID")
)

person_counts = (
    aami_df.groupby(["Person_ID", "AAMI_Label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_ORDER, fill_value=0)
    .reset_index()
    .sort_values("Person_ID")
)

record_counts.to_csv(
    OUTPUT_DIR / "m0_class_counts_by_record.csv",
    index=False
)

person_counts.to_csv(
    OUTPUT_DIR / "m0_class_counts_by_person.csv",
    index=False
)

print("CLASS COUNTS BY RECORD")
display(record_counts)

print("CLASS COUNTS BY PERSON")
display(person_counts)

print("Number of people contributing to each class:")

for c in CLASS_ORDER:
    people = int((person_counts[c] > 0).sum())
    beats = int(person_counts[c].sum())

    print(
        f"{c}: {beats:,} beats across {people} people"
    )

CLASS COUNTS BY RECORD


AAMI_Label,Record_ID,Person_ID,N,S,V,F,Q
0,100,person_100,2237,33,1,0,0
1,101,person_101,1859,3,0,0,2
2,103,person_103,2082,2,0,0,0
3,105,person_105,2526,0,41,0,5
4,106,person_106,1507,0,520,0,0
5,107,person_107,0,0,59,0,2077
6,108,person_108,1739,4,17,2,0
7,109,person_109,2491,0,38,2,0
8,111,person_111,2123,0,1,0,0
9,112,person_112,2537,2,0,0,0


CLASS COUNTS BY PERSON


AAMI_Label,Person_ID,N,S,V,F,Q
0,person_100,2237,33,1,0,0
1,person_101,1859,3,0,0,2
2,person_103,2082,2,0,0,0
3,person_105,2526,0,41,0,5
4,person_106,1507,0,520,0,0
5,person_107,0,0,59,0,2077
6,person_108,1739,4,17,2,0
7,person_109,2491,0,38,2,0
8,person_111,2123,0,1,0,0
9,person_112,2537,2,0,0,0


Number of people contributing to each class:
N: 90,341 beats across 44 people
S: 2,781 beats across 31 people
V: 7,229 beats across 34 people
F: 802 beats across 16 people
Q: 3,894 beats across 7 people


In [4]:
y = aami_df["AAMI_Label"].to_numpy()
groups = aami_df["Person_ID"].to_numpy()
record_ids = aami_df["Record_ID"].to_numpy()
row_ids = aami_df["Row_Index"].to_numpy()

split_indices = {}

person_membership = []
row_membership = []
fold_support = []

for repeat in range(1, N_REPEATS + 1):

    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED + repeat - 1
    )

    folds = []
    assigned = np.zeros(len(aami_df), dtype=np.int8)

    for fold, (train_idx, val_idx) in enumerate(
        splitter.split(
            np.zeros(len(y)),
            y,
            groups=groups
        ),
        start=1
    ):

        train_people = set(groups[train_idx])
        val_people = set(groups[val_idx])

        assert train_people.isdisjoint(val_people)

        assert set(record_ids[train_idx]).isdisjoint(
            set(record_ids[val_idx])
        )

        assigned[val_idx] += 1
        folds.append((train_idx, val_idx))

        for person in sorted(set(groups[val_idx])):
            person_membership.append({
                "Repeat": repeat,
                "Fold": fold,
                "Person_ID": person
            })

        counts = (
            pd.Series(y[val_idx])
            .value_counts()
            .reindex(CLASS_ORDER, fill_value=0)
        )

        for c in CLASS_ORDER:
            fold_support.append({
                "Repeat": repeat,
                "Fold": fold,
                "Class": c,
                "Validation_Beats": int(counts[c])
            })

    assert np.all(assigned == 1)

    split_indices[repeat] = folds

    fold_for_row = np.empty(
        len(aami_df),
        dtype=np.int8
    )

    for fold, (_, val_idx) in enumerate(folds, start=1):
        fold_for_row[val_idx] = fold

    row_membership.append(pd.DataFrame({
        "Row_Index": row_ids,
        "Repeat": repeat,
        "Fold": fold_for_row
    }))

person_fold = pd.DataFrame(person_membership).sort_values(
    ["Repeat", "Fold", "Person_ID"]
)

row_fold = pd.concat(
    row_membership,
    ignore_index=True
)

support = pd.DataFrame(fold_support)

assert len(person_fold) == EXPECTED_PEOPLE * N_REPEATS
assert not person_fold.duplicated(
    ["Repeat", "Person_ID"]
).any()

person_fold.to_csv(
    OUTPUT_DIR / "m0_person_fold_membership.csv",
    index=False
)

row_fold.to_csv(
    OUTPUT_DIR / "m0_row_fold_membership.csv.gz",
    index=False,
    compression="gzip"
)

support.to_csv(
    OUTPUT_DIR / "m0_validation_class_support_by_fold.csv",
    index=False
)

fingerprint = hashlib.sha256(
    person_fold.sort_values(
        ["Repeat", "Person_ID"]
    ).to_csv(
        index=False,
        lineterminator="\n"
    ).encode()
).hexdigest()

(OUTPUT_DIR / "m0_split_fingerprint.txt").write_text(
    fingerprint + "\n"
)

display(person_fold)
display(support)

print("Split fingerprint:", fingerprint)
print("PASS: no person-level or record-level leakage.")

,Repeat,Fold,Person_ID
0,1,1,person_106
1,1,1,person_115
2,1,1,person_121
3,1,1,person_123
4,1,1,person_205
...,...,...,...
220,5,5,person_208
221,5,5,person_221
222,5,5,person_223
223,5,5,person_231


,Repeat,Fold,Class,Validation_Beats
0,1,1,N,16815
1,1,1,S,1530
2,1,1,V,1010
3,1,1,F,383
4,1,1,Q,0
...,...,...,...,...
120,5,5,N,15693
121,5,5,S,1555
122,5,5,V,1924
123,5,5,F,386


Split fingerprint: 5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9
PASS: no person-level or record-level leakage.


In [5]:
if ACCOUNT_ROLE == "A_REFERENCE_MODEL":

    wave = aami_df[
        [f"MLII_{i}" for i in range(201)]
    ].to_numpy(dtype=np.float32)

    means = wave.mean(axis=1, keepdims=True)
    stds = wave.std(axis=1, keepdims=True)
    stds[stds == 0] = 1.0

    normalized = (wave - means) / stds

    d1 = np.diff(normalized, axis=1)
    d2 = np.diff(normalized, n=2, axis=1)

    q25, q50, q75 = np.percentile(
        normalized,
        [25, 50, 75],
        axis=1
    )

    X = np.column_stack([
        d1.mean(axis=1),
        d1.std(axis=1),
        d1.max(axis=1),
        d1.min(axis=1),
        np.abs(d1).mean(axis=1),
        np.square(d1).mean(axis=1),
        d2.std(axis=1),
        np.abs(d2).mean(axis=1),
        q50,
        q75 - q25,
        q25,
        q75
    ]).astype(np.float32)

    EXPERIMENT = "REFERENCE_MODEL"
    TREES = 200
    MAX_FEATURES = "log2"

else:

    X = aami_df[SIGNAL_COLS].to_numpy(
        dtype=np.float32
    )

    EXPERIMENT = "BASE_RAW_402"
    TREES = 100
    MAX_FEATURES = "sqrt"

assert X.shape == (
    len(aami_df),
    12 if EXPERIMENT == "REFERENCE_MODEL" else 402
)

assert np.isfinite(X).all()

print("Experiment:", EXPERIMENT)
print("Feature matrix:", X.shape)
print("Trees:", TREES)

Experiment: REFERENCE_MODEL
Feature matrix: (105047, 12)
Trees: 200


In [6]:
prediction_parts = []

for repeat, folds in split_indices.items():

    for fold, (train_idx, val_idx) in enumerate(
        folds,
        start=1
    ):

        model = RandomForestClassifier(
            n_estimators=TREES,
            max_features=MAX_FEATURES,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=SEED,
            n_jobs=-1,
            class_weight=None
        )

        model.fit(X[train_idx], y[train_idx])

        y_pred = model.predict(X[val_idx])

        raw_prob = model.predict_proba(X[val_idx])

        probabilities = np.zeros(
            (len(val_idx), len(CLASS_ORDER)),
            dtype=np.float32
        )

        for j, c in enumerate(model.classes_):
            probabilities[
                :, CLASS_ORDER.index(str(c))
            ] = raw_prob[:, j]

        part = pd.DataFrame({
            "Experiment": EXPERIMENT,
            "Repeat": repeat,
            "Fold": fold,
            "Row_Index": row_ids[val_idx],
            "Record_ID": record_ids[val_idx],
            "Person_ID": groups[val_idx],
            "True_Label": y[val_idx],
            "Predicted_Label": y_pred
        })

        for j, c in enumerate(CLASS_ORDER):
            part[f"Prob_{c}"] = probabilities[:, j]

        prediction_parts.append(part)

        print(
            f"Completed {EXPERIMENT}: "
            f"repeat {repeat}, fold {fold}"
        )

predictions = pd.concat(
    prediction_parts,
    ignore_index=True
)

assert len(predictions) == EXPECTED_BEATS * N_REPEATS

assert not predictions.duplicated(
    ["Repeat", "Row_Index"]
).any()

assert predictions.groupby(
    "Repeat"
)["Row_Index"].nunique().eq(EXPECTED_BEATS).all()

pred_path = OUTPUT_DIR / (
    f"m0_oof_predictions_{EXPERIMENT}.csv.gz"
)

predictions.to_csv(
    pred_path,
    index=False,
    compression="gzip"
)

print("OOF predictions saved:", pred_path)

Completed REFERENCE_MODEL: repeat 1, fold 1
Completed REFERENCE_MODEL: repeat 1, fold 2
Completed REFERENCE_MODEL: repeat 1, fold 3
Completed REFERENCE_MODEL: repeat 1, fold 4
Completed REFERENCE_MODEL: repeat 1, fold 5
Completed REFERENCE_MODEL: repeat 2, fold 1
Completed REFERENCE_MODEL: repeat 2, fold 2
Completed REFERENCE_MODEL: repeat 2, fold 3
Completed REFERENCE_MODEL: repeat 2, fold 4
Completed REFERENCE_MODEL: repeat 2, fold 5
Completed REFERENCE_MODEL: repeat 3, fold 1
Completed REFERENCE_MODEL: repeat 3, fold 2
Completed REFERENCE_MODEL: repeat 3, fold 3
Completed REFERENCE_MODEL: repeat 3, fold 4
Completed REFERENCE_MODEL: repeat 3, fold 5
Completed REFERENCE_MODEL: repeat 4, fold 1
Completed REFERENCE_MODEL: repeat 4, fold 2
Completed REFERENCE_MODEL: repeat 4, fold 3
Completed REFERENCE_MODEL: repeat 4, fold 4
Completed REFERENCE_MODEL: repeat 4, fold 5
Completed REFERENCE_MODEL: repeat 5, fold 1
Completed REFERENCE_MODEL: repeat 5, fold 2
Completed REFERENCE_MODEL: repea

In [7]:
metric_rows = []
class_rows = []
confusion_rows = []

for repeat in range(1, N_REPEATS + 1):

    part = predictions[
        predictions["Repeat"] == repeat
    ]

    yt = part["True_Label"].to_numpy()
    yp = part["Predicted_Label"].to_numpy()

    precision, recall, f1, support_values = (
        precision_recall_fscore_support(
            yt,
            yp,
            labels=CLASS_ORDER,
            zero_division=0
        )
    )

    non_f = part[
        part["True_Label"].isin(FOUR_CLASS_ORDER)
    ]

    metrics = {
        "Experiment": EXPERIMENT,
        "Repeat": repeat,
        "Macro_F1_5_Class": f1_score(
            yt, yp,
            labels=CLASS_ORDER,
            average="macro",
            zero_division=0
        ),
        "Macro_F1_4_Class_N_S_V_Q": f1_score(
            non_f["True_Label"],
            non_f["Predicted_Label"],
            labels=FOUR_CLASS_ORDER,
            average="macro",
            zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            yt, yp
        ),
        "Accuracy": accuracy_score(yt, yp)
    }

    metric_rows.append(metrics)

    class_rows.append(pd.DataFrame({
        "Experiment": EXPERIMENT,
        "Repeat": repeat,
        "Class": CLASS_ORDER,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Support": support_values
    }))

    cm = confusion_matrix(
        yt, yp,
        labels=CLASS_ORDER
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual_{c}" for c in CLASS_ORDER],
        columns=[f"Predicted_{c}" for c in CLASS_ORDER]
    )

    cm_df["Repeat"] = repeat
    confusion_rows.append(cm_df)

repeat_metrics = pd.DataFrame(metric_rows)
per_class = pd.concat(class_rows, ignore_index=True)
confusions = pd.concat(confusion_rows)

headline = pd.DataFrame([{
    "Experiment": EXPERIMENT,
    "Headline_Macro_F1_5_Class": (
        repeat_metrics["Macro_F1_5_Class"].mean()
    ),
    "Macro_F1_4_Class_N_S_V_Q": (
        repeat_metrics["Macro_F1_4_Class_N_S_V_Q"].mean()
    ),
    "Balanced_Accuracy": (
        repeat_metrics["Balanced_Accuracy"].mean()
    ),
    "Accuracy": repeat_metrics["Accuracy"].mean()
}])

display(headline)
display(repeat_metrics)
display(per_class)
display(confusions)

repeat_metrics.to_csv(
    OUTPUT_DIR / f"m0_repeat_metrics_{EXPERIMENT}.csv",
    index=False
)

per_class.to_csv(
    OUTPUT_DIR / f"m0_per_class_metrics_{EXPERIMENT}.csv",
    index=False
)

confusions.to_csv(
    OUTPUT_DIR / f"m0_confusion_matrices_{EXPERIMENT}.csv"
    , index=True
)

headline.to_csv(
    OUTPUT_DIR / f"m0_headline_summary_{EXPERIMENT}.csv",
    index=False
)

,Experiment,Headline_Macro_F1_5_Class,Macro_F1_4_Class_N_S_V_Q,Balanced_Accuracy,Accuracy
0,REFERENCE_MODEL,0.433125,0.54409,0.465256,0.893948


,Experiment,Repeat,Macro_F1_5_Class,Macro_F1_4_Class_N_S_V_Q,Balanced_Accuracy,Accuracy
0,REFERENCE_MODEL,1,0.518730,0.651630,0.535893,0.913477
1,REFERENCE_MODEL,2,0.315283,0.396376,0.363569,0.887336
2,REFERENCE_MODEL,3,0.516553,0.648760,0.538987,0.907384
3,REFERENCE_MODEL,4,0.309936,0.389738,0.360398,0.866536
4,REFERENCE_MODEL,5,0.505120,0.633946,0.527434,0.895009


,Experiment,Repeat,Class,Precision,Recall,F1,Support
0,REFERENCE_MODEL,1,N,0.948738,0.956509,0.952608,90341
1,REFERENCE_MODEL,1,S,0.017241,0.000360,0.000704,2781
2,REFERENCE_MODEL,1,V,0.609514,0.850740,0.710203,7229
3,REFERENCE_MODEL,1,F,0.000000,0.000000,0.000000,802
4,REFERENCE_MODEL,1,Q,0.996770,0.871854,0.930137,3894
5,REFERENCE_MODEL,2,N,0.949350,0.963505,0.956375,90341
6,REFERENCE_MODEL,2,S,0.000000,0.000000,0.000000,2781
7,REFERENCE_MODEL,2,V,0.485170,0.853092,0.618556,7229
8,REFERENCE_MODEL,2,F,0.001832,0.001247,0.001484,802
9,REFERENCE_MODEL,2,Q,0.000000,0.000000,0.000000,3894


,Predicted_N,Predicted_S,Predicted_V,Predicted_F,Predicted_Q,Repeat
Actual_N,86412,55,3502,362,10,1
Actual_S,2629,1,136,15,0,1
Actual_V,1041,2,6150,35,1,1
Actual_F,564,0,238,0,0,1
Actual_Q,435,0,64,0,3395,1
Actual_N,87044,87,2716,488,6,2
Actual_S,2633,0,142,5,1,2
Actual_V,1002,7,6167,52,1,2
Actual_F,558,0,243,1,0,2
Actual_Q,451,0,3443,0,0,2


In [8]:
def record_bootstrap_ci(
    pred,
    metric="Macro_F1_5_Class",
    n_boot=BOOTSTRAP_SAMPLES,
    seed=BOOTSTRAP_SEED
):

    rng = np.random.default_rng(seed)

    record_list = np.array(
        sorted(pred["Record_ID"].unique())
    )

    blocks = {
        (repeat, record): group
        for (repeat, record), group
        in pred.groupby(["Repeat", "Record_ID"])
    }

    repeats = sorted(pred["Repeat"].unique())
    bootstrap_scores = []

    for _ in range(n_boot):

        sampled_records = rng.choice(
            record_list,
            size=len(record_list),
            replace=True
        )

        repeat_scores = []

        for repeat in repeats:

            sample = pd.concat([
                blocks[(repeat, record)]
                for record in sampled_records
            ])

            yt = sample["True_Label"].to_numpy()
            yp = sample["Predicted_Label"].to_numpy()

            if metric == "Macro_F1_5_Class":

                score = f1_score(
                    yt, yp,
                    labels=CLASS_ORDER,
                    average="macro",
                    zero_division=0
                )

            else:

                keep = np.isin(
                    yt,
                    FOUR_CLASS_ORDER
                )

                score = f1_score(
                    yt[keep],
                    yp[keep],
                    labels=FOUR_CLASS_ORDER,
                    average="macro",
                    zero_division=0
                )

            repeat_scores.append(score)

        bootstrap_scores.append(
            np.mean(repeat_scores)
        )

    point_scores = []

    for repeat in repeats:

        sample = pred[pred["Repeat"] == repeat]

        yt = sample["True_Label"].to_numpy()
        yp = sample["Predicted_Label"].to_numpy()

        if metric == "Macro_F1_5_Class":

            score = f1_score(
                yt, yp,
                labels=CLASS_ORDER,
                average="macro",
                zero_division=0
            )

        else:

            keep = np.isin(
                yt,
                FOUR_CLASS_ORDER
            )

            score = f1_score(
                yt[keep],
                yp[keep],
                labels=FOUR_CLASS_ORDER,
                average="macro",
                zero_division=0
            )

        point_scores.append(score)

    lower, upper = np.quantile(
        bootstrap_scores,
        [0.025, 0.975]
    )

    return {
        "Metric": metric,
        "Point_Estimate": np.mean(point_scores),
        "CI_95_Lower": lower,
        "CI_95_Upper": upper,
        "Bootstrap_Samples": n_boot,
        "Resampling_Unit": "Record_ID"
    }


ci_results = pd.DataFrame([
    record_bootstrap_ci(
        predictions,
        "Macro_F1_5_Class"
    ),
    record_bootstrap_ci(
        predictions,
        "Macro_F1_4_Class_N_S_V_Q"
    )
])

ci_results.insert(0, "Experiment", EXPERIMENT)

display(ci_results)

ci_results.to_csv(
    OUTPUT_DIR / f"m0_record_bootstrap_ci_{EXPERIMENT}.csv",
    index=False
)

,Experiment,Metric,Point_Estimate,CI_95_Lower,CI_95_Upper,Bootstrap_Samples,Resampling_Unit
0,REFERENCE_MODEL,Macro_F1_5_Class,0.433125,0.315710,0.465527,1000,Record_ID
1,REFERENCE_MODEL,Macro_F1_4_Class_N_S_V_Q,0.544090,0.397131,0.586404,1000,Record_ID


In [9]:
total_counts = (
    aami_df["AAMI_Label"]
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
)

demo_impact = person_counts.copy()

for c in CLASS_ORDER:

    demo_impact[
        f"{c}_Share_Of_Class_Percent"
    ] = (
        100 * demo_impact[c] / total_counts[c]
    )

demo_impact["Total_Beats"] = (
    demo_impact[CLASS_ORDER].sum(axis=1)
)

demo_impact["Rare_Class_Beats_S_V_F_Q"] = (
    demo_impact[["S", "V", "F", "Q"]].sum(axis=1)
)

demo_impact = demo_impact.sort_values(
    ["Rare_Class_Beats_S_V_F_Q", "Total_Beats"],
    ascending=False
)

display(demo_impact)

demo_impact.to_csv(
    OUTPUT_DIR / "m0_demo_person_impact.csv",
    index=False
)

print(
    "No people have been removed. "
    "Demo selection requires review and approval."
)

AAMI_Label,Person_ID,N,S,V,F,Q,N_Share_Of_Class_Percent,S_Share_Of_Class_Percent,V_Share_Of_Class_Percent,F_Share_Of_Class_Percent,Q_Share_Of_Class_Percent,Total_Beats,Rare_Class_Beats_S_V_F_Q
5,person_107,0,0,59,0,2077,0.000000,0.000000,0.816157,0.000000,53.338469,2136,2136
33,person_217,244,0,162,0,1802,0.270088,0.000000,2.240974,0.000000,46.276323,2208,1964
42,person_232,398,1382,0,0,0,0.440553,49.694355,0.000000,0.000000,0.000000,1780,1382
26,person_208,1585,2,992,372,2,1.754464,0.071917,13.722507,46.384040,0.051361,2953,1368
21,person_200,1742,30,826,2,0,1.928250,1.078749,11.426200,0.249377,0.000000,2600,858
43,person_233,2229,7,830,11,0,2.467318,0.251708,11.481533,1.371571,0.000000,3077,848
30,person_213,2639,28,220,362,0,2.921154,1.006832,3.043298,45.137157,0.000000,3249,610
38,person_223,2045,73,473,14,0,2.263646,2.624955,6.543090,1.745636,0.000000,2605,560
4,person_106,1507,0,520,0,0,1.668124,0.000000,7.193249,0.000000,0.000000,2027,520
23,person_203,2528,2,444,1,4,2.798286,0.071917,6.141928,0.124688,0.102722,2979,451


No people have been removed. Demo selection requires review and approval.


In [10]:
checks = {
    "105047_valid_beats": len(aami_df) == 105047,
    "expected_class_counts": observed_counts == EXPECTED_COUNTS,
    "46_records_45_people": (
        aami_df["Record_ID"].nunique() == 46
        and aami_df["Person_ID"].nunique() == 45
    ),
    "201_202_same_person": person_201[0] == person_202[0],
    "25_outer_folds": len(split_indices) * N_SPLITS == 25,
    "OOF_predictions_complete": (
        len(predictions) == EXPECTED_BEATS * N_REPEATS
    ),
    "finite_features": bool(np.isfinite(X).all())
}

check_table = pd.DataFrame([
    {
        "Check": name,
        "Result": "PASS" if passed else "FAIL"
    }
    for name, passed in checks.items()
])

display(check_table)

check_table.to_csv(
    OUTPUT_DIR / f"m0_final_checks_{EXPERIMENT}.csv",
    index=False
)

if not all(checks.values()):
    raise AssertionError(
        "M0 checks failed. Stop and investigate."
    )

manifest = {
    "notebook": "01_M0_MEASUREMENT_AND_SPLIT_PREPARATION",
    "account_role": ACCOUNT_ROLE,
    "experiment": EXPERIMENT,
    "valid_beats": len(aami_df),
    "records": int(aami_df["Record_ID"].nunique()),
    "people": int(aami_df["Person_ID"].nunique()),
    "class_counts": observed_counts,
    "seed": SEED,
    "folds": N_SPLITS,
    "repeats": N_REPEATS,
    "split_fingerprint": fingerprint,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "bootstrap_unit": "Record_ID",
    "demo_people_removed": []
}

(
    OUTPUT_DIR / f"m0_manifest_{EXPERIMENT}.json"
).write_text(
    json.dumps(manifest, indent=2)
)

print("M0 baseline completed:", EXPERIMENT)
print(
    "Headline Macro-F1:",
    round(
        float(headline.iloc[0]["Headline_Macro_F1_5_Class"]),
        6
    )
)
print("Split fingerprint:", fingerprint)
print("Outputs saved in:", OUTPUT_DIR)
print("STOP: wait for demo-person review before excluding anyone.")

,Check,Result
0,105047_valid_beats,PASS
1,expected_class_counts,PASS
2,46_records_45_people,PASS
3,201_202_same_person,PASS
4,25_outer_folds,PASS
5,OOF_predictions_complete,PASS
6,finite_features,PASS


M0 baseline completed: REFERENCE_MODEL
Headline Macro-F1: 0.433125
Split fingerprint: 5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9
Outputs saved in: /kaggle/working/ECG_Project/01_M0_Measurement
STOP: wait for demo-person review before excluding anyone.
